# Vehicle Service Management and Maintenance Analysis
### VCET - Department of Information Technology
**Course:** 25ITC31 – Foundations of Data Science  
**Activity:** Project-Based Assessment (Scenario 16)  
**Register Nos:** 16, 46, 76, 106

---

## 1. Import Required Libraries and Setup Environment

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 2. Ingest Dataset from Relational Database (Unit 2 & 3)

In [ ]:
db_path = os.path.join("..", "database", "vehicle_service.db")
conn = sqlite3.connect(db_path)

query = """
SELECT 
    c.customer_id, c.name AS customer_name, c.phone,
    v.vehicle_id, v.reg_number, v.model, v.vehicle_type, v.manufacture_year,
    a.appointment_id, a.appointment_date, a.service_type, a.status AS appointment_status,
    sr.service_id, sr.technician_name, sr.work_performed, sr.spare_parts_cost, sr.labor_charges,
    b.bill_id, b.bill_date, b.parts_cost, b.labor_cost, b.tax_amount, b.discount_amount, b.total_amount,
    b.payment_status, b.payment_method
FROM customers c
JOIN vehicles v ON c.customer_id = v.customer_id
JOIN appointments a ON v.vehicle_id = a.vehicle_id
JOIN service_records sr ON a.appointment_id = sr.appointment_id
JOIN bills b ON sr.service_id = b.service_id
"""

df = pd.read_sql_query(query, conn)
conn.close()
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

## 3. Data Cleaning, Transformation & Feature Engineering (Unit 4)

In [ ]:
df['appointment_date'] = pd.to_datetime(df['appointment_date'])
df['bill_date'] = pd.to_datetime(df['bill_date'])
df['service_month'] = df['bill_date'].dt.strftime('%Y-%m')
df['vehicle_age'] = 2026 - df['manufacture_year']
df['subtotal'] = df['parts_cost'] + df['labor_cost']
df['parts_share_pct'] = (df['parts_cost'] / df['subtotal']) * 100

print("Missing Values:\n", df.isnull().sum().loc[lambda x: x > 0])
df.info()

## 4. Descriptive Statistics & Exploration (Unit 1 & 5)

In [ ]:
stat_cols = ['parts_cost', 'labor_cost', 'tax_amount', 'total_amount', 'vehicle_age']
df[stat_cols].describe().round(2)

## 5. Visualizations & Pattern Discovery (Unit 5)
### 5.1 Service Request Frequency by Category

In [ ]:
plt.figure(figsize=(10, 5))
service_counts = df['service_type'].value_counts()
sns.barplot(x=service_counts.index, y=service_counts.values, hue=service_counts.index, palette='viridis', legend=False)
plt.title("Service Center Workload by Category", fontweight='bold')
plt.xlabel("Service Type")
plt.ylabel("Count of Services")
plt.xticks(rotation=20)
plt.show()

### 5.2 Monthly Revenue and Job Count Trends

In [ ]:
monthly = df.groupby('service_month').agg({'total_amount': 'sum', 'service_id': 'count'}).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(monthly['service_month'], monthly['total_amount'], color='#3498db', alpha=0.7, label='Revenue (Rs)')
ax1.set_ylabel('Total Revenue (INR ₹)', color='#2980b9')

ax2 = ax1.twinx()
ax2.plot(monthly['service_month'], monthly['service_id'], color='#e74c3c', marker='o', linewidth=2.5, label='Jobs Count')
ax2.set_ylabel('Jobs Completed', color='#e74c3c')
ax2.grid(False)
plt.title("Monthly Revenue & Service Volume Trends (2026)", fontweight='bold')
plt.show()

### 5.3 Vehicle Type Distribution (Pie Chart)

In [ ]:
plt.figure(figsize=(7, 7))
type_counts = df['vehicle_type'].value_counts()
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('Set2', len(type_counts)))
plt.title("Distribution of Serviced Vehicle Types", fontweight='bold')
plt.show()

### 5.4 Correlation Heatmap of Financial Attributes

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[['parts_cost', 'labor_cost', 'tax_amount', 'discount_amount', 'total_amount', 'vehicle_age']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=1)
plt.title("Correlation Heatmap", fontweight='bold')
plt.show()